<a href="https://www.kaggle.com/code/n210799shaikshaheen/major-project?scriptVersionId=368294272" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

**Models Training**
1. ResNet50 — resnet50

2. DenseNet121 — densenet121

3. EfficientNet-B0 — efficientnet_b0

4. MobileNetV2 — mobilenet_v2

In [ ]:
!ls /kaggle/input/datasets/n210799shaikshaheen/pneumonia-project

In [ ]:
!cp -r /kaggle/input/datasets/n210799shaikshaheen/pneumonia-project/Pneumonia_Adversarial_Robustness /kaggle/working/
%cd /kaggle/working/Pneumonia_Adversarial_Robustness

In [ ]:
!mkdir -p /kaggle/working/Pneumonia_Adversarial_Robustness
!cp -r /kaggle/input/datasets/n210799shaikshaheen/pneumonia-project/* /kaggle/working/Pneumonia_Adversarial_Robustness/
%cd /kaggle/working/Pneumonia_Adversarial_Robustness

In [ ]:
!ls    # you should see main.py, app.py, config.py, attacks, models, ...

In [ ]:
!ls /kaggle/input/datasets
!find /kaggle/input -maxdepth 4 -type d -name train

In [ ]:
import os
p = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
print(os.path.exists(p), os.listdir(p) if os.path.exists(p) else "NOT FOUND")

In [ ]:
!find /kaggle/input -maxdepth 5 -type d -name train

In [ ]:
import os, glob
base = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia"
hits = glob.glob(base + "/**/train", recursive=True)
print(hits)
data_dir = os.path.dirname(hits[0])      # folder that contains train/ val/ test/
print("using:", data_dir, os.listdir(data_dir))
os.environ["PNEUMONIA_DATA_DIR"] = data_dir

In [ ]:
%cd /kaggle/working/Pneumonia_Adversarial_Robustness
!ls

In [ ]:
%cd /kaggle/working/Pneumonia_Adversarial_Robustness
!sed -i 's/^NUM_WORKERS = .*/NUM_WORKERS = 4/' config.py
!ls

In [ ]:
!python main.py --demo


# **Training and Validation**

In [ ]:
!rm -rf models/checkpoints experiment_cache results
!python main.py --train-only

In [ ]:
import os

folder = "/kaggle/working/Pneumonia_Adversarial_Robustness/results"

print("Folder exists:", os.path.exists(folder))
print("Files:", os.listdir(folder) if os.path.exists(folder) else "Not found")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/results",
    "zip",
    "/kaggle/working/Pneumonia_Adversarial_Robustness/results"
)

In [ ]:

import os
import pandas as pd
from IPython.display import display

# Results folder
folder = "/kaggle/working/Pneumonia_Adversarial_Robustness/results"

# Find and display every CSV file
csv_files = []

for root, dirs, files in os.walk(folder):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("Total CSV files:", len(csv_files))

for path in sorted(csv_files):
    print("\n" + "=" * 150)
    print("FILE:", os.path.basename(path))
    print("=" * 150)

    df = pd.read_csv(path)

    # Convert metric columns to percentages
    percentage_columns = [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "mean_confidence"
    ]

    for col in percentage_columns:
        if col in df.columns:
            df[col] = (df[col] * 100).round(2).astype(str) + "%"

    display(df)


In [ ]:
import os
import json
import matplotlib.pyplot as plt

folder = "/kaggle/working/Pneumonia_Adversarial_Robustness/results/training_history"

models = [
    "densenet121",
    "efficientnet_b0",
    "mobilenet_v2",
    "resnet50"
]

for model in models:
    path = os.path.join(folder, model + ".json")

    with open(path, "r") as f:
        data = json.load(f)

    epochs = data["epoch"]

    # Create two plots side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Training Loss
    axes[0].plot(epochs, data["train_loss"], marker="o")
    axes[0].set_title(model + " - Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True)

    # Training and Validation Accuracy
    axes[1].plot(epochs, data["train_acc"], marker="o",
                 label="Training Accuracy")
    axes[1].plot(epochs, data["val_acc"], marker="o",
                 label="Validation Accuracy")
    axes[1].set_title(model + " - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True)

    plt.suptitle(model.upper() + " Training Performance", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

path = "/kaggle/working/Pneumonia_Adversarial_Robustness/results/clean_results.csv"

df = pd.read_csv(path)

metrics = ["accuracy", "precision", "recall", "f1"]

df.set_index("display_name")[metrics].mul(100).plot(
    kind="bar",
    figsize=(11, 5)
)

plt.title("Clean Performance Comparison of Models")
plt.ylabel("Percentage (%)")
plt.xlabel("Model")
plt.ylim(0, 110)
plt.xticks(rotation=0)
plt.legend(title="Metrics")
plt.grid(axis="y")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import ast
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

path = "/kaggle/working/Pneumonia_Adversarial_Robustness/results/clean_results.csv"

df = pd.read_csv(path)

# Create 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Flatten axes for easy indexing
axes = axes.flatten()

for i, (_, row) in enumerate(df.iterrows()):

    cm = np.array(ast.literal_eval(row["confusion_matrix"]))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Normal", "Pneumonia"]
    )

    disp.plot(
        cmap="Blues",
        values_format="d",
        ax=axes[i],
        colorbar=False
    )

    axes[i].set_title(row["display_name"] + " Confusion Matrix")

plt.tight_layout()
plt.show()




## Adversarial Attack Methods

1. **FGSM** – Fast Gradient Sign Method
2. **PGD** – Projected Gradient Descent
3. **DeepFool** – DeepFool Attack
4. **CW** – Carlini & Wagner Attack

## Defense Methods

1. **JPEG** – JPEG Compression
2. **Gaussian** – Gaussian Noise Filtering
3. **Median** – Median Filtering
4. **BitDepth** – Bit-Depth Reduction
5. **Smoothing** – Image Smoothing
6. **AdvTrain** – Adversarial Training



In [ ]:
%cd /kaggle/working/Pneumonia_Adversarial_Robustness

N_SAMPLES = 200
EPSILON = 0.02

# 1. ResNet50

In [ ]:

!python main.py --models resnet50 \
    --attacks FGSM PGD DeepFool CW \
    --defenses JPEG Gaussian Median BitDepth Smoothing AdvTrain \
    --epsilon 0.02 --n-samples 200

In [ ]:

import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display


def display_attack_defense_images(
    model="resnet50",
    project="/kaggle/working/Pneumonia_Adversarial_Robustness"
):

    grid_folder = os.path.join(
        project, "outputs", "comparison_grids", model
    )

    if not os.path.exists(grid_folder):
        print(f"No comparison folder found for {model}")
        return

    attacks = ["FGSM", "PGD", "DeepFool", "CW"]

    for attack in attacks:

        print(f"\n{'='*20} {attack} {'='*20}")

        attack_folder = os.path.join(grid_folder, attack)

        # 1. Original vs Attacked
        files = glob.glob(
            os.path.join(
                attack_folder,
                "*_clean_vs_attacked.png"
            )
        )

        if files:
            latest = max(files, key=os.path.getmtime)

            print("Original vs Attacked:")
            display(Image.open(latest))
        else:
            print("No Original vs Attacked image found.")

        # 2. Attacked vs Defended
        defense_files = glob.glob(
            os.path.join(
                attack_folder,
                "*_attacked_vs_defended.png"
            )
        )

        if defense_files:
            latest_defense = max(
                defense_files,
                key=os.path.getmtime
            )

            print("Attacked vs Defended:")
            display(Image.open(latest_defense))
        else:
            print("No Attacked vs Defended image found.")


# Run for ResNet50
display_attack_defense_images("resnet50")




# Results displaying function 

1. Attack results table.
2. Attack accuracy graph.
3. Attack Success Rate (ASR) graph.
4. Defense results table.
5. Defense accuracy comparison graph.
6. Clean confusion matrix.
7. Save graphs and CSV results.


In [ ]:

import os
import ast
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay

BASE_DIR = Path("/kaggle/working/Pneumonia_Adversarial_Robustness")


def display_model_results(model="resnet50"):

    model = model.lower()
    output_dir = BASE_DIR / "outputs" / f"{model}_final_results"
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 75)
    print(f"       ADVERSARIAL ROBUSTNESS RESULTS: {model.upper()}")
    print("=" * 75)

    # ---------------------------------------------------------
    # 1. SEARCH FOR CSV FILES
    # ---------------------------------------------------------

    csv_files = []

    for file in BASE_DIR.rglob("*.csv"):
        relative_parts = [part.lower() for part in file.relative_to(BASE_DIR).parts]

        # Avoid reading generated outputs and cached experiment files
        if any(
            part in ["outputs", "experiment_cache", "demo_run"]
            for part in relative_parts
        ):
            continue

        csv_files.append(file)

    attack_frames = []
    defense_frames = []
    clean_frames = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df.columns = df.columns.str.strip()

            # Filter for the requested model
            if "model" in df.columns:
                df = df[
                    df["model"].astype(str).str.lower() == model
                ].copy()

            if df.empty:
                continue

            df["_source_file"] = str(file)

            # Defense CSV
            if {
                "attack", "defense", "defended_acc",
                "attacked_acc", "recovery"
            }.issubset(df.columns):
                defense_frames.append(df)

            # Attack CSV
            elif {
                "attack", "attacked_acc", "clean_acc"
            }.issubset(df.columns):
                attack_frames.append(df)

            # Clean evaluation CSV
            if "confusion_matrix" in df.columns:
                clean_frames.append(df)

        except Exception as e:
            print(f"Skipped {file.name}: {e}")

    print("\n1. CSV FILES FOUND")
    print("Attack CSV files:", len(attack_frames))
    print("Defense CSV files:", len(defense_frames))
    print("Clean result CSV files:", len(clean_frames))

    # ---------------------------------------------------------
    # 2. HELPER: NORMALIZE METRICS
    # ---------------------------------------------------------

    def to_percentage(series):
        values = pd.to_numeric(series, errors="coerce")
        return values.apply(
            lambda x: x * 100 if pd.notna(x) and abs(x) <= 1 else x
        )

    # ---------------------------------------------------------
    # 3. ATTACK RESULTS
    # ---------------------------------------------------------

    attacks = pd.DataFrame()

    if attack_frames:
        attacks = pd.concat(attack_frames, ignore_index=True)

        for col in ["clean_acc", "attacked_acc", "attack_success_rate"]:
            if col in attacks.columns:
                attacks[col] = to_percentage(attacks[col])

        # Keep one row per attack
        attacks = attacks.sort_values("_source_file")
        attacks = attacks.drop_duplicates(
            subset=["attack"], keep="last"
        )

    elif defense_frames:
        # Defense CSV also contains clean and attacked accuracy
        # Use one row per attack to build the attack comparison.
        all_defenses = pd.concat(defense_frames, ignore_index=True)

        attacks = all_defenses.sort_values("_source_file").drop_duplicates(
            subset=["attack"], keep="last"
        ).copy()

        attacks["clean_acc"] = to_percentage(attacks["clean_acc"])
        attacks["attacked_acc"] = to_percentage(attacks["attacked_acc"])

    print("\n2. ATTACK RESULTS")

    if not attacks.empty:
        attack_columns = [
            col for col in [
                "attack", "tag", "clean_acc",
                "attacked_acc", "attack_success_rate", "n"
            ]
            if col in attacks.columns
        ]

        attack_table = attacks[attack_columns].copy()

        for col in ["clean_acc", "attacked_acc", "attack_success_rate"]:
            if col in attack_table.columns:
                attack_table[col] = attack_table[col].round(2)

        display(attack_table)

        attack_table.to_csv(
            output_dir / "attack_results.csv", index=False
        )

        # Attack accuracy graph
        if {"attack", "clean_acc", "attacked_acc"}.issubset(attack_table.columns):

            plot_data = attack_table.set_index("attack")[
                ["clean_acc", "attacked_acc"]
            ]

            ax = plot_data.plot(
                kind="bar",
                figsize=(10, 5),
                width=0.75
            )

            ax.set_title(f"{model.upper()} - Clean vs Attacked Accuracy")
            ax.set_ylabel("Accuracy (%)")
            ax.set_xlabel("Attack")
            ax.set_ylim(0, 100)
            ax.legend(["Clean Accuracy", "Attacked Accuracy"])
            ax.grid(axis="y", alpha=0.3)
            plt.xticks(rotation=0)
            plt.tight_layout()
            plt.savefig(
                output_dir / "attack_accuracy.png",
                dpi=300,
                bbox_inches="tight"
            )
            plt.show()

        # Baseline attack success rate graph, if available
        if (
            "attack_success_rate" in attack_table.columns
            and attack_table["attack_success_rate"].notna().any()
        ):
            asr_data = attack_table.dropna(
                subset=["attack_success_rate"]
            )

            plt.figure(figsize=(9, 5))
            plt.bar(
                asr_data["attack"],
                asr_data["attack_success_rate"]
            )
            plt.title(f"{model.upper()} - Attack Success Rate")
            plt.ylabel("Attack Success Rate (%)")
            plt.xlabel("Attack")
            plt.ylim(0, 100)
            plt.grid(axis="y", alpha=0.3)
            plt.tight_layout()
            plt.savefig(
                output_dir / "attack_success_rate.png",
                dpi=300,
                bbox_inches="tight"
            )
            plt.show()
        else:
            print(
                "Baseline attack success rate is not available "
                "in the attack CSV. It will not be guessed."
            )

    else:
        print("No attack results found.")

    # ---------------------------------------------------------
    # 4. DEFENSE RESULTS
    # ---------------------------------------------------------

    print("\n3. DEFENSE RESULTS")

    defenses = pd.DataFrame()

    if defense_frames:
        defenses = pd.concat(defense_frames, ignore_index=True)

        # Convert metrics to percentages independently.
        for col in [
            "clean_acc",
            "attacked_acc",
            "defended_acc",
            "recovery",
            "attack_success_rate_after_defense"
        ]:
            if col in defenses.columns:
                defenses[col] = to_percentage(defenses[col])

        # Remove exact duplicates, then retain one row per attack/defense.
        defenses = defenses.drop_duplicates()

        defenses = defenses.sort_values("_source_file")
        defenses = defenses.drop_duplicates(
            subset=["attack", "defense"], keep="last"
        )

        defense_columns = [
            col for col in [
                "attack", "defense", "clean_acc",
                "attacked_acc", "defended_acc", "recovery",
                "attack_success_rate_after_defense"
            ]
            if col in defenses.columns
        ]

        defense_table = defenses[defense_columns].copy()

        for col in defense_columns:
            if col not in ["attack", "defense"]:
                defense_table[col] = defense_table[col].round(2)

        display(defense_table)

        defense_table.to_csv(
            output_dir / "defense_results.csv", index=False
        )

        # Defense accuracy graph
        pivot_accuracy = defenses.pivot_table(
            index="attack",
            columns="defense",
            values="defended_acc",
            aggfunc="mean"
        )

        ax = pivot_accuracy.plot(
            kind="bar",
            figsize=(12, 6),
            width=0.8
        )

        ax.set_title(f"{model.upper()} - Defense Accuracy Comparison")
        ax.set_ylabel("Defended Accuracy (%)")
        ax.set_xlabel("Attack")
        ax.set_ylim(0, 100)
        ax.legend(title="Defense", bbox_to_anchor=(1.02, 1), loc="upper left")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(
            output_dir / "defense_accuracy.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

        # Attack success rate after defense
        if (
            "attack_success_rate_after_defense" in defenses.columns
            and defenses["attack_success_rate_after_defense"].notna().any()
        ):
            pivot_asr = defenses.pivot_table(
                index="attack",
                columns="defense",
                values="attack_success_rate_after_defense",
                aggfunc="mean"
            )

            ax = pivot_asr.plot(
                kind="bar",
                figsize=(12, 6),
                width=0.8
            )

            ax.set_title(
                f"{model.upper()} - Attack Success Rate After Defense"
            )
            ax.set_ylabel("Attack Success Rate (%)")
            ax.set_xlabel("Attack")
            ax.set_ylim(0, 100)
            ax.legend(
                title="Defense",
                bbox_to_anchor=(1.02, 1),
                loc="upper left"
            )
            ax.grid(axis="y", alpha=0.3)
            plt.xticks(rotation=0)
            plt.tight_layout()
            plt.savefig(
                output_dir / "asr_after_defense.png",
                dpi=300,
                bbox_inches="tight"
            )
            plt.show()

        # Best defense for each attack
        best_defenses = (
            defenses.sort_values("defended_acc", ascending=False)
            .drop_duplicates(subset=["attack"])
        )

        print("\n4. BEST DEFENSE FOR EACH ATTACK")

        best_columns = [
            col for col in [
                "attack", "defense", "defended_acc", "recovery"
            ]
            if col in best_defenses.columns
        ]

        display(
            best_defenses[best_columns]
            .sort_values("attack")
            .reset_index(drop=True)
        )

        best_defenses[best_columns].to_csv(
            output_dir / "best_defenses.csv", index=False
        )

    else:
        print("No defense results found.")

    # ---------------------------------------------------------
    # 5. CLEAN CONFUSION MATRIX
    # ---------------------------------------------------------

    print("\n5. CLEAN CONFUSION MATRIX")

    confusion_matrix = None

    for df in clean_frames:
        if "confusion_matrix" not in df.columns:
            continue

        for value in df["confusion_matrix"].dropna():
            try:
                if isinstance(value, str):
                    try:
                        matrix = ast.literal_eval(value)
                    except Exception:
                        matrix = json.loads(value)
                else:
                    matrix = value

                matrix = np.asarray(matrix, dtype=int)

                if matrix.shape == (2, 2):
                    confusion_matrix = matrix
                    break

            except Exception:
                continue

        if confusion_matrix is not None:
            break

    if confusion_matrix is not None:

        print(confusion_matrix)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=confusion_matrix,
            display_labels=["Class 0", "Class 1"]
        )

        fig, ax = plt.subplots(figsize=(6, 5))
        disp.plot(
            cmap="Blues",
            values_format="d",
            ax=ax,
            colorbar=False
        )

        ax.set_title(f"{model.upper()} - Clean Confusion Matrix")
        plt.tight_layout()
        plt.savefig(
            output_dir / "clean_confusion_matrix.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

        np.savetxt(
            output_dir / "clean_confusion_matrix.csv",
            confusion_matrix,
            delimiter=",",
            fmt="%d"
        )

        total = confusion_matrix.sum()
        accuracy = np.trace(confusion_matrix) / total * 100

        print(f"Clean test accuracy from confusion matrix: {accuracy:.2f}%")
        print(f"Total test images: {total}")

    else:
        print("No valid 2x2 clean confusion matrix found.")

    # ---------------------------------------------------------
    # 6. FINISH
    # ---------------------------------------------------------

    print("\n" + "=" * 75)
    print("RESULTS GENERATED SUCCESSFULLY")
    print("=" * 75)
    print("Saved in:", output_dir)

    return {
        "attacks": attacks,
        "defenses": defenses,
        "confusion_matrix": confusion_matrix,
        "output_dir": str(output_dir)
    }


# ---------------------------------------------------------
# RUN FOR ONE MODEL
# ---------------------------------------------------------

resnet_results = display_model_results("resnet50")


In [ ]:

import os
import numpy as np
import pandas as pd
from IPython.display import display


def display_defense_tables(results, model="resnet50"):

    # Get defense results
    df = results["defenses"].copy()

    if df.empty:
        print("No defense results found.")
        return None

    # Remove duplicate attack-defense combinations
    df = df.drop_duplicates(
        subset=["attack", "defense"],
        keep="first"
    ).copy()

    # Normalize metrics into percentages
    for col in ["defended_acc", "recovery"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

        df[col] = np.where(
            df[col].abs() <= 1,
            df[col] * 100,
            df[col]
        )

    # Define display order
    attack_order = ["FGSM", "PGD", "DeepFool", "CW"]

    defense_order = [
        "JPEG", "Gaussian", "Median",
        "BitDepth", "Smoothing", "AdvTrain"
    ]

    # Output directory
    output_dir = results["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    # 1. DEFENDED ACCURACY
    # Rows = Defenses, Columns = Attacks
    accuracy_table = df.pivot(
        index="defense",
        columns="attack",
        values="defended_acc"
    ).reindex(
        index=defense_order,
        columns=attack_order
    ).round(2)

    print(f"\n{model.upper()} - DEFENDED ACCURACY (%)")
    display(accuracy_table)

    accuracy_table.to_csv(
        os.path.join(
            output_dir,
            f"{model}_defended_accuracy.csv"
        )
    )

    # 2. RECOVERY
    # Rows = Defenses, Columns = Attacks
    recovery_table = df.pivot(
        index="defense",
        columns="attack",
        values="recovery"
    ).reindex(
        index=defense_order,
        columns=attack_order
    ).round(2)

    print(f"\n{model.upper()} - RECOVERY (%)")
    display(recovery_table)

    recovery_table.to_csv(
        os.path.join(
            output_dir,
            f"{model}_recovery.csv"
        )
    )

    # 3. BEST DEFENSE PER ATTACK
    best = (
        df.sort_values(
            "defended_acc",
            ascending=False
        )
        .drop_duplicates(subset="attack")
        .set_index("attack")
        .reindex(attack_order)
        .reset_index()
    )

    best_table = best[
        ["attack", "defense", "defended_acc", "recovery"]
    ].copy()

    best_table.columns = [
        "Attack",
        "Best Defense",
        "Defended Accuracy (%)",
        "Recovery (%)"
    ]

    print(f"\n{model.upper()} - BEST DEFENSE")
    display(best_table.round(2))

    best_table.to_csv(
        os.path.join(
            output_dir,
            f"{model}_best_defenses.csv"
        ),
        index=False
    )

    print("\nAll tables saved successfully.")
    print("Saved location:", output_dir)

    return {
        "accuracy": accuracy_table,
        "recovery": recovery_table,
        "best_defenses": best_table,
        "output_dir": output_dir
    }


# Run for ResNet50
resnet_defense_tables = display_defense_tables(
    resnet_results,
    model="resnet50"
)


# 2. DenseNet121

In [ ]:
!python main.py --models densenet121 \
    --attacks FGSM PGD DeepFool CW \
    --defenses JPEG Gaussian Median BitDepth Smoothing AdvTrain \
    --epsilon 0.02 --n-samples 200

In [ ]:
# Run for DenseNet121
display_attack_defense_images("densenet121")

In [ ]:
densenet_results = display_model_results("densenet121")

In [ ]:
densenet_defense_tables = display_defense_tables(
    densenet_results, "densenet121"
)

# 3. EfficientNetB0

In [ ]:
!python main.py --models efficientnet_b0 \
    --attacks FGSM PGD DeepFool CW \
    --defenses JPEG Gaussian Median BitDepth Smoothing AdvTrain \
    --epsilon 0.02 --n-samples 200

In [ ]:
# Run for EfficientNet-B0
display_attack_defense_images("efficientnet_b0")

In [ ]:
efficientnet_results = display_model_results("efficientnet_b0")

In [ ]:
efficientnet_defense_tables = display_defense_tables(
    efficientnet_results, "efficientnet_b0"
)
  

# 4. MobileNetV2

In [ ]:
!python main.py --models mobilenet_v2 \
    --attacks FGSM PGD DeepFool CW \
    --defenses JPEG Gaussian Median BitDepth Smoothing AdvTrain \
    --epsilon 0.02 --n-samples 200

In [ ]:
# Run for MobileNetV2
display_attack_defense_images("mobilenet_v2")

In [ ]:
mobilenet_results = display_model_results("mobilenet_v2")

In [ ]:
mobilenet_defense_tables = display_defense_tables(
    mobilenet_results, "mobilenet_v2"
)

# COMPARISON ACROSS MODELS

In [ ]:
!zip -rq results_bundle.zip models/checkpoints outputs results

In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "/kaggle/working/Pneumonia_Adversarial_Robustness"

output_dir = os.path.join(BASE_DIR, "outputs", "all_models_line_comparison")
os.makedirs(output_dir, exist_ok=True)

all_results = {
    "ResNet50": resnet_results,
    "DenseNet121": densenet_results,
    "EfficientNet-B0": efficientnet_results,
    "MobileNetV2": mobilenet_results
}

attack_order = ["FGSM", "PGD", "DeepFool", "CW"]

def get_numeric(df, column):
    if df.empty or column not in df.columns:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[column], errors="coerce")

# Collect results
attack_frames = []
defense_frames = []
clean_frames = []

for model_name, results in all_results.items():

    attacks = results["attacks"].copy()
    defenses = results["defenses"].copy()
    cm = results.get("confusion_matrix")

    if not attacks.empty:
        attacks["model"] = model_name
        attack_frames.append(attacks)

    if not defenses.empty:
        defenses["model"] = model_name
        defense_frames.append(defenses)

    if cm is not None:
        total = cm.sum()
        if total > 0:
            clean_frames.append({
                "model": model_name,
                "clean_accuracy": cm.trace() / total * 100
            })

attack_df = (
    pd.concat(attack_frames, ignore_index=True)
    if attack_frames else pd.DataFrame()
)

defense_df = (
    pd.concat(defense_frames, ignore_index=True)
    if defense_frames else pd.DataFrame()
)

clean_df = pd.DataFrame(clean_frames)

models = list(all_results.keys())

# --------------------------------------------------
# 1. CLEAN TEST ACCURACY - ALL MODELS
# --------------------------------------------------

if not clean_df.empty:
    plt.figure(figsize=(9, 5))

    plt.plot(
        clean_df["model"],
        clean_df["clean_accuracy"],
        marker="o",
        linewidth=2
    )

    plt.title("Clean Test Accuracy - All Models")
    plt.xlabel("Models")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 105)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "01_clean_accuracy.png"), dpi=300)
    plt.show()
    plt.close()

# --------------------------------------------------
# 2. CLEAN VS ATTACKED ACCURACY - ALL MODELS
# --------------------------------------------------

if not attack_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for model in models:
        temp = attack_df[attack_df["model"] == model].copy()
        temp["attack"] = pd.Categorical(
            temp["attack"], attack_order, ordered=True
        )
        temp = temp.sort_values("attack")

        axes[0].plot(
            temp["attack"].astype(str),
            get_numeric(temp, "clean_acc"),
            marker="o",
            linewidth=2,
            label=model
        )

        axes[1].plot(
            temp["attack"].astype(str),
            get_numeric(temp, "attacked_acc"),
            marker="o",
            linewidth=2,
            label=model
        )

    axes[0].set_title("Clean Accuracy Comparison")
    axes[1].set_title("Attacked Accuracy Comparison")

    for ax in axes:
        ax.set_xlabel("Attack")
        ax.set_ylabel("Accuracy (%)")
        ax.set_ylim(0, 105)
        ax.grid(True)
        ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "02_clean_vs_attacked.png"), dpi=300)
    plt.show()
    plt.close()

# --------------------------------------------------
# 3. DEFENDED ACCURACY - ALL MODELS
# --------------------------------------------------

if not defense_df.empty:
    plt.figure(figsize=(13, 7))

    for model in models:
        temp = defense_df[defense_df["model"] == model].copy()

        grouped = temp.groupby("attack", observed=False)[
            "defended_acc"
        ].mean()

        grouped = grouped.reindex(attack_order)

        plt.plot(
            grouped.index,
            grouped.values,
            marker="o",
            linewidth=2,
            label=model
        )

    plt.title("Overall Defended Accuracy - All Models")
    plt.xlabel("Attack")
    plt.ylabel("Average Defended Accuracy (%)")
    plt.ylim(0, 105)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "03_defended_accuracy.png"), dpi=300)
    plt.show()
    plt.close()

# --------------------------------------------------
# 4. DEFENSE-WISE COMPARISON - ALL MODELS
# --------------------------------------------------

if not defense_df.empty:
    defenses = [
        "JPEG", "Gaussian", "Median",
        "BitDepth", "Smoothing", "AdvTrain"
    ]

    for attack in attack_order:
        temp_attack = defense_df[
            defense_df["attack"] == attack
        ]

        if temp_attack.empty:
            continue

        plt.figure(figsize=(12, 6))

        for model in models:
            temp = temp_attack[
                temp_attack["model"] == model
            ]

            grouped = temp.groupby("defense")[
                "defended_acc"
            ].mean().reindex(defenses)

            plt.plot(
                grouped.index,
                grouped.values,
                marker="o",
                linewidth=2,
                label=model
            )

        plt.title(f"{attack} - Defense Accuracy Comparison")
        plt.xlabel("Defense Methods")
        plt.ylabel("Defended Accuracy (%)")
        plt.ylim(0, 105)
        plt.xticks(rotation=20)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            os.path.join(output_dir, f"04_{attack.lower()}_defenses.png"),
            dpi=300
        )
        plt.show()
        plt.close()

# --------------------------------------------------
# 5. RECOVERY COMPARISON - ALL MODELS
# --------------------------------------------------

if not defense_df.empty and "recovery" in defense_df.columns:

    plt.figure(figsize=(12, 6))

    for model in models:
        temp = defense_df[defense_df["model"] == model]

        grouped = temp.groupby("attack")["recovery"].mean()
        grouped = grouped.reindex(attack_order)

        plt.plot(
            grouped.index,
            grouped.values,
            marker="o",
            linewidth=2,
            label=model
        )

    plt.title("Average Accuracy Recovery - All Models")
    plt.xlabel("Attack")
    plt.ylabel("Recovery (%)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "05_recovery.png"), dpi=300)
    plt.show()
    plt.close()

# --------------------------------------------------
# 6. ASR AFTER DEFENSE - ALL MODELS
# --------------------------------------------------

if not defense_df.empty and "attack_success_rate_after_defense" in defense_df.columns:

    plt.figure(figsize=(12, 6))

    for model in models:
        temp = defense_df[defense_df["model"] == model]

        grouped = temp.groupby("attack")[
            "attack_success_rate_after_defense"
        ].mean()

        grouped = grouped.reindex(attack_order)

        plt.plot(
            grouped.index,
            grouped.values,
            marker="o",
            linewidth=2,
            label=model
        )

    plt.title("Average ASR After Defense - All Models")
    plt.xlabel("Attack")
    plt.ylabel("ASR (%)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "06_asr.png"), dpi=300)
    plt.show()
    plt.close()

print("All model comparison plots generated.")
print("Saved at:", output_dir)


In [ ]:

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from IPython.display import display

# --------------------------------------------------
# 1. SETTINGS
# --------------------------------------------------

BASE_DIR = "/kaggle/working/Pneumonia_Adversarial_Robustness"

OUTPUT_DIR = os.path.join(
    BASE_DIR, "outputs", "interactive_comparison"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_results = {
    "ResNet50": resnet_results,
    "DenseNet121": densenet_results,
    "EfficientNet-B0": efficientnet_results,
    "MobileNetV2": mobilenet_results
}

models = list(all_results.keys())

attacks_order = ["FGSM", "DeepFool", "CW", "PGD"]

defenses_order = [
    "JPEG", "Gaussian", "Median",
    "BitDepth", "Smoothing", "AdvTrain"
]

# --------------------------------------------------
# 2. COLLECT RESULTS
# --------------------------------------------------

attack_frames = []
defense_frames = []
clean_records = []

for model, results in all_results.items():

    attacks = results["attacks"].copy()
    defenses = results["defenses"].copy()
    cm = results.get("confusion_matrix")

    if not attacks.empty:
        attacks["model"] = model
        attack_frames.append(attacks)

    if not defenses.empty:
        defenses["model"] = model
        defense_frames.append(defenses)

    if cm is not None:
        cm = np.asarray(cm)
        if cm.sum() > 0:
            clean_records.append({
                "model": model,
                "clean_accuracy": np.trace(cm) / cm.sum() * 100
            })

attack_df = (
    pd.concat(attack_frames, ignore_index=True)
    if attack_frames else pd.DataFrame()
)

defense_df = (
    pd.concat(defense_frames, ignore_index=True)
    if defense_frames else pd.DataFrame()
)

clean_df = pd.DataFrame(clean_records)


# --------------------------------------------------
# 3. NORMALIZE ATTACK AND DEFENSE NAMES
# --------------------------------------------------

def normalize_attack(value):

    value = str(value).lower().strip()

    if "fgsm" in value:
        return "FGSM"
    if "pgd" in value:
        return "PGD"
    if "deepfool" in value:
        return "DeepFool"
    if value.startswith("cw") or "carlini" in value:
        return "CW"

    return value


if not attack_df.empty:
    attack_df["attack_clean"] = attack_df["attack"].apply(normalize_attack)

if not defense_df.empty:
    defense_df["attack_clean"] = defense_df["attack"].apply(normalize_attack)


def convert_percentage(series):

    series = pd.to_numeric(series, errors="coerce")

    if series.notna().any() and series.max() <= 1:
        series = series * 100

    return series


if not attack_df.empty:
    for col in ["clean_acc", "attacked_acc"]:
        if col in attack_df.columns:
            attack_df[col] = convert_percentage(attack_df[col])

if not defense_df.empty:
    for col in ["attacked_acc", "defended_acc", "recovery"]:
        if col in defense_df.columns:
            defense_df[col] = convert_percentage(defense_df[col])

    defense_df["defense_clean"] = (
        defense_df["defense"].astype(str).str.strip().str.lower()
    )

if not clean_df.empty:
    clean_df["clean_accuracy"] = convert_percentage(
        clean_df["clean_accuracy"]
    )


# --------------------------------------------------
# GRAPH 1: CLEAN MODEL ACCURACY - INTERACTIVE BAR
# --------------------------------------------------

fig1 = go.Figure()

for _, row in clean_df.iterrows():

    fig1.add_trace(go.Bar(
        x=[row["model"]],
        y=[row["clean_accuracy"]],
        name=row["model"],
        text=[f'{row["clean_accuracy"]:.2f}%'],
        textposition="outside",
        hovertemplate=(
            "<b>Model:</b> %{x}<br>"
            "<b>Clean Accuracy:</b> %{y:.2f}%"
            "<extra></extra>"
        )
    ))

fig1.update_layout(
    title="Clean Model Accuracy Comparison",
    xaxis_title="Models",
    yaxis_title="Clean Accuracy (%)",
    yaxis_range=[0, 110],
    template="plotly_white",
    barmode="group",
    hovermode="closest"
)

fig1.show()

fig1.write_html(
    os.path.join(OUTPUT_DIR, "01_clean_accuracy.html")
)


# --------------------------------------------------
# GRAPH 2: CLEAN VS ATTACKED ACCURACY
# FOUR INTERACTIVE MODEL LINES
# --------------------------------------------------

fig2 = go.Figure()

x_labels = ["Clean", "FGSM", "DeepFool", "CW"]

for model in models:

    values = []

    clean_value = clean_df.loc[
        clean_df["model"] == model,
        "clean_accuracy"
    ]

    values.append(
        clean_value.iloc[0] if not clean_value.empty else np.nan
    )

    temp = attack_df[attack_df["model"] == model]

    for attack in ["FGSM", "DeepFool", "CW"]:

        result = temp.loc[
            temp["attack_clean"] == attack,
            "attacked_acc"
        ]

        values.append(
            result.mean() if not result.empty else np.nan
        )

    fig2.add_trace(go.Scatter(
        x=x_labels,
        y=values,
        mode="lines+markers",
        name=model,
        connectgaps=False,
        line=dict(width=3),
        marker=dict(size=9),
        hovertemplate=(
            "<b>Model:</b> " + model + "<br>"
            "<b>Condition:</b> %{x}<br>"
            "<b>Accuracy:</b> %{y:.2f}%"
            "<extra></extra>"
        )
    ))

fig2.update_layout(
    title="Clean vs Adversarial Attack Accuracy",
    xaxis_title="Attack Type",
    yaxis_title="Accuracy (%)",
    yaxis_range=[0, 105],
    template="plotly_white",
    hovermode="x unified"
)

fig2.show()

fig2.write_html(
    os.path.join(OUTPUT_DIR, "02_accuracy_under_attack.html")
)


# --------------------------------------------------
# GRAPH 3: NO DEFENSE VS ALL DEFENSES
# SEVEN INTERACTIVE LINES
# --------------------------------------------------

fig3 = go.Figure()

attack_names = ["FGSM", "PGD", "DeepFool", "CW"]

# No defense
no_defense_values = []

for attack in attack_names:

    temp = attack_df[
        attack_df["attack_clean"] == attack
    ]

    no_defense_values.append(
        temp["attacked_acc"].mean()
        if not temp.empty else np.nan
    )

fig3.add_trace(go.Scatter(
    x=attack_names,
    y=no_defense_values,
    mode="lines+markers",
    name="No Defence",
    line=dict(width=4, dash="dash"),
    marker=dict(size=10),
    hovertemplate=(
        "<b>Method:</b> No Defence<br>"
        "<b>Attack:</b> %{x}<br>"
        "<b>Accuracy:</b> %{y:.2f}%"
        "<extra></extra>"
    )
))

# Six defense lines
for defense in defenses_order:

    values = []

    for attack in attack_names:

        temp = defense_df[
            (defense_df["attack_clean"] == attack) &
            (defense_df["defense_clean"] == defense.lower())
        ]

        values.append(
            temp["defended_acc"].mean()
            if not temp.empty else np.nan
        )

    if not all(pd.isna(values)):

        fig3.add_trace(go.Scatter(
            x=attack_names,
            y=values,
            mode="lines+markers",
            name=defense,
            line=dict(width=2.5),
            marker=dict(size=8),
            connectgaps=False,
            hovertemplate=(
                "<b>Defense:</b> " + defense + "<br>"
                "<b>Attack:</b> %{x}<br>"
                "<b>Defended Accuracy:</b> %{y:.2f}%"
                "<extra></extra>"
            )
        ))

fig3.update_layout(
    title="Accuracy Under Attacks: No Defence vs All Defenses",
    xaxis_title="Adversarial Attacks",
    yaxis_title="Accuracy (%)",
    yaxis_range=[0, 105],
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        title="Defense Methods",
        orientation="v"
    )
)

fig3.show()

fig3.write_html(
    os.path.join(OUTPUT_DIR, "03_all_defenses.html")
)


# --------------------------------------------------
# GRAPH 4: CLEAN VS ATTACKED VS DEFENDED
# INTERACTIVE GROUPED BAR CHART
# --------------------------------------------------

comparison = []

for model in models:

    clean_value = clean_df.loc[
        clean_df["model"] == model,
        "clean_accuracy"
    ]

    attacked_value = attack_df.loc[
        attack_df["model"] == model,
        "attacked_acc"
    ]

    defended_value = defense_df.loc[
        defense_df["model"] == model,
        "defended_acc"
    ]

    comparison.append({
        "model": model,
        "Clean": clean_value.mean() if not clean_value.empty else np.nan,
        "Attacked": attacked_value.mean() if not attacked_value.empty else np.nan,
        "Defended": defended_value.mean() if not defended_value.empty else np.nan
    })

comparison_df = pd.DataFrame(comparison)

fig4 = go.Figure()

for condition in ["Clean", "Attacked", "Defended"]:

    fig4.add_trace(go.Bar(
        x=comparison_df["model"],
        y=comparison_df[condition],
        name=condition,
        text=[
            f"{v:.2f}%" if pd.notna(v) else "N/A"
            for v in comparison_df[condition]
        ],
        textposition="outside",
        hovertemplate=(
            "<b>Model:</b> %{x}<br>"
            "<b>Condition:</b> " + condition + "<br>"
            "<b>Accuracy:</b> %{y:.2f}%"
            "<extra></extra>"
        )
    ))

fig4.update_layout(
    title="Clean vs Attacked vs Defended Accuracy",
    xaxis_title="Models",
    yaxis_title="Accuracy (%)",
    yaxis_range=[0, 115],
    barmode="group",
    template="plotly_white",
    hovermode="x unified"
)

fig4.show()

fig4.write_html(
    os.path.join(OUTPUT_DIR, "04_clean_attacked_defended.html")
)

# --------------------------------------------------
# SAVE DATA AND FINISH
# --------------------------------------------------

comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_comparison.csv"),
    index=False
)

print("All 4 interactive graphs generated!")
print("Output folder:", OUTPUT_DIR)


In [ ]:

import shutil

shutil.make_archive(
    "/kaggle/working/Pneumonia_Project",
    "zip",
    "/kaggle/working/Pneumonia_Adversarial_Robustness"
)

print("ZIP file created successfully!")
